# Analyse Baseline Autoencoder Model: Failure Cases

In [ ]:
import os
import sys
import numpy as np
import torch
import torch.nn as nn
from torch.autograd import Variable
from torch.utils.data import DataLoader
from torchvision.transforms import Compose
from tqdm import tqdm
import mlflow.pytorch
import imgaug.augmenters as iaa
from pprint import pprint
import pandas as pd
import cv2

import mlflow

import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append('../../../')

from src.data import TrainValTestSplitter, MURASubset
from src.data.transforms import GrayScale, Padding, Resize, HistEqualisation, MinMaxNormalization, ToTensor
from src.features.augmentation import Augmentation
#from src.models.autoencoders import BaselineAutoencoder
from src.models.torchsummary import summary
from src import DATA_PATH, MODELS_DIR, MLFLOW_TRACKING_URI
from src.models import BaselineAutoencoder

## Setup

Set up general parameters and get model from MLFlow

In [ ]:
size_sample = 20 # set how many image to be analysed

In [ ]:
client = mlflow.tracking.MlflowClient(MLFLOW_TRACKING_URI)
client.list_experiments()

In [ ]:
run_id = '0ca304cc755f4203a5c6576edb14e4b9'
experiment = client.get_experiment('1')
path = f'{experiment.artifact_location}/{run_id}/artifacts/baseline_autoencoder/data/model.pth'
path

Set up parameters and initiate model and dataset.

In [ ]:
data_path = f'{DATA_PATH}/XR_HAND_CROPPED'


splitter = TrainValTestSplitter(data_path) 

run_params = {
    'batch_size': 64,
    'image_resolution': (512, 512),
    'num_epochs': 500,
    'batch_normalisation': True,
    'pipeline': {
        'hist_equalisation': True,
        'cropped': False,
    }
}

composed_transforms = Compose([GrayScale(),
                               Padding(),
                               Resize(run_params['image_resolution']),
                               HistEqualisation(active=run_params['pipeline']['hist_equalisation']),
                               MinMaxNormalization(),
                               ToTensor()])

validation = MURASubset(filenames=splitter.data_val.path, true_labels=splitter.data_val.label,
                        patients=splitter.data_val.patient, transform=composed_transforms)


num_workers = 6

device = "cuda"
model = torch.load(path)

outer_loss = nn.MSELoss(reduction='none')
model.eval().to(device)

In [ ]:
composed_transforms = Compose([GrayScale(),
                               Padding(max_shape=(750, 750)),  # max_shape - max size of image after augmentation
                               Resize(run_params['image_resolution']),
                               HistEqualisation(active=run_params['pipeline']['hist_equalisation']),
                               MinMaxNormalization(),
                               ToTensor()])

# Dataset loaders
print(f'\nDATA SPLIT:')
splitter = TrainValTestSplitter(path_to_data=data_path)
validation = MURASubset(filenames=splitter.data_val.path, true_labels=splitter.data_val.label,
                        patients=splitter.data_val.patient, transform=composed_transforms)

val_loader = DataLoader(validation, batch_size=run_params['batch_size'], shuffle=True, num_workers=num_workers)

## Run evaluation

In [ ]:
val_metrics = model.evaluate(val_loader, 'validation', 
                             outer_loss, device, log_to_mlflow=False)

In [ ]:
masked_loss_on_val = False
# Evaluation mode
model.eval()
with torch.no_grad():
    scores = []
    true_labels = []
    file_path = []
    for batch_data in tqdm(val_loader, desc='type', total=len(val_loader)):
        # Format input batch
        inp = batch_data['image'].to(device)
        mask = batch_data['mask'].to(device)

        # Forward pass
        output = model(inp)
        loss = outer_loss(output, inp, mask) if masked_loss_on_val else outer_loss(output, inp)

        # Scores, based on MSE - higher MSE correspond to abnormal image
        if masked_loss_on_val:
            sum_loss = loss.to('cpu').numpy().sum(axis=(1, 2, 3))
            sum_mask = mask.to('cpu').numpy().sum(axis=(1, 2, 3))
            score = sum_loss / sum_mask
        else:
            score = loss.to('cpu').numpy().mean(axis=(1, 2, 3))

        scores.extend(score)
        true_labels.extend(batch_data['label'].numpy())
        file_path.extend(batch_data['filename'])

val_metrics = {'loss': scores, 'true_label': true_labels, 'file_path': file_path}

In [ ]:
print(val_metrics)

### Set up data frame for evaluation

This evaluation is based on validation set and follows the idea of ROC AUC. High MSE should be predicated as 1 and low as 0.

In [ ]:
data = pd.DataFrame.from_dict(val_metrics)


In [ ]:
print(data)

### Analyse False Negative

In [ ]:
false_negative = data[data['true_label'] == 1].nsmallest(size_sample, 'loss')
false_negative.style.set_properties(subset=['file_path'], **{'width': '300px'})

In [ ]:
files_false_negative_path = false_negative['file_path']
files_false_negative = MURASubset(filenames=files_false_negative_path, transform=composed_transforms)

In [ ]:
fig, ax = plt.subplots(nrows=size_sample, ncols=2, figsize=(20, 8*size_sample))

for i in range(size_sample):
    image_inp = files_false_negative[i]['image'][None,:,:,:]
    inp = image_inp.to(device)
    output = model(inp)
    
    ax[i, 0].imshow(image_inp.numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)
    ax[i, 1].imshow(output.cpu().detach().numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)

Reasons for false negative:
  - relatively small images with a lot of black space
  - relatively little noise
  - common hands except for the arm without hand (why can the model reconstruct this so good?) and one with only parts of the hand

In [ ]:
fig, ax = plt.subplots(nrows=size_sample, ncols=1, figsize=(20, 8*size_sample))

for i in range(size_sample):
    image_inp = files_false_negative[i]['image'][None,:,:,:]
    inp = image_inp.to(device)
    output = model(inp)
    image_substract = output.cpu().detach().numpy()[0, 0, :, :] - image_inp.numpy()[0, 0, :, :]
    ax[i].imshow(image_substract, cmap='gray', vmin=0, vmax=1)

### Analyse False positive

In [ ]:
false_positive = data[data['true_label'] == 0].nlargest(size_sample, 'loss')
false_positive.style.set_properties(subset=['file_path'], **{'width': '300px'})

In [ ]:
files_false_positive_path = false_positive['file_path']
files_false_positive = MURASubset(filenames=files_false_positive_path, transform=composed_transforms)

In [ ]:
fig, ax = plt.subplots(nrows=size_sample, ncols=2, figsize=(20, 8*size_sample))

for i in range(size_sample):
    image_inp = files_false_positive[i]['image'][None,:,:,:]
    inp = image_inp.to(device)
    output = model(inp)
    
    ax[i, 0].imshow(image_inp.numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)
    ax[i, 1].imshow(output.cpu().detach().numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)

Reasons for false positive:
  - a lot of noise around the hand &rarr; masking might help

In [ ]:
fig, ax = plt.subplots(nrows=size_sample, ncols=1, figsize=(20, 8*size_sample))

for i in range(size_sample):
    image_inp = files_false_negative[i]['image'][None,:,:,:]
    inp = image_inp.to(device)
    output = model(inp)
    image_substract = output.cpu().detach().numpy()[0, 0, :, :] - image_inp.numpy()[0, 0, :, :]
    ax[i].imshow(image_substract, cmap='gray', vmin=0, vmax=1)

### Analyse True Negative

In [ ]:
true_negative = data[data['true_label'] == 0].nsmallest(size_sample, 'loss')
true_negative.style.set_properties(subset=['file_path'], **{'width': '300px'})

In [ ]:
files_true_negative_path = true_negative['file_path']
files_true_negative = MURASubset(filenames=files_true_negative_path, transform=composed_transforms)

In [ ]:
fig, ax = plt.subplots(nrows=size_sample, ncols=2, figsize=(20, 8*size_sample))

for i in range(size_sample):
    image_inp = files_true_negative[i]['image'][None,:,:,:]
    inp = image_inp.to(device)
    output = model(inp)
    
    ax[i, 0].imshow(image_inp.numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)
    ax[i, 1].imshow(output.cpu().detach().numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)

### Analyse True Positive

In [ ]:
true_positive = data[data['true_label'] == 1].nlargest(size_sample, 'loss')
true_positive.style.set_properties(subset=['file_path'], **{'width': '300px'})

In [ ]:
files_true_positive_path = true_positive['file_path']
files_true_positive = MURASubset(filenames=files_true_positive_path, transform=composed_transforms)

In [ ]:
fig, ax = plt.subplots(nrows=size_sample, ncols=2, figsize=(20, 8*size_sample))

for i in range(size_sample):
    image_inp = files_true_positive[i]['image'][None,:,:,:]
    inp = image_inp.to(device)
    output = model(inp)
    
    ax[i, 0].imshow(image_inp.numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)
    ax[i, 1].imshow(output.cpu().detach().numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)

Reasions for true positive:
  - a lot of noise &rarr; model can't reconstruct noise &rarr; classified as positve see following plot

In [ ]:
fig, ax = plt.subplots(nrows=size_sample, ncols=1, figsize=(20, 8*size_sample))

for i in range(size_sample):
    image_inp = files_true_positive[i]['image'][None,:,:,:]
    inp = image_inp.to(device)
    output = model(inp)
    image_substract = output.cpu().detach().numpy()[0, 0, :, :] - image_inp.numpy()[0, 0, :, :]
    ax[i].imshow(image_substract, cmap='gray', vmin=0, vmax=1)